In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd() if '__file__' not in globals() else Path(__file__).resolve().parent
_ROOT = _HERE.parents[1]
sys.path.insert(0, str(_HERE))

import numpy as np
import mujoco
import mujoco.viewer
from numpy.linalg import inv
from scipy.linalg import solve_continuous_are
MJCF = str(_ROOT / 'models' / 'scenes' / 'cartpole' / 'scene.xml')

##### cartpole의 비선형 동역학

In [ ]:
class CartPoleDynamics():
    def __init__(self, m_c=1.0, m_p=1.0, L=1.0):
        self.m_c = m_c
        self.m_p = m_p
        self.L   = L
        self.g   = 9.81

    def _M(self, theta):
        return np.array([
            [self.m_c + self.m_p, self.m_p * self.L * np.cos(theta)],
            [self.m_p * self.L * np.cos(theta), self.m_p * self.L**2]
        ])

    def _C(self, theta, dtheta):
        return np.array([
            [0, -self.m_p * self.L * dtheta**2 * np.sin(theta)],
            [0, 0]
        ])

    def _G(self, theta):
        return np.array([
            [0],
            [self.m_p * self.L * self.g * np.sin(theta)]
        ])

    def continuous_dynamics(self, x, u):
        cart_pos, pole_angle, cart_vel, pole_vel = x
        dq = np.array([[cart_vel], [pole_vel]])
        B  = np.array([[1], [0]])

        M = self._M(pole_angle)
        C = self._C(pole_angle, pole_vel)
        G = self._G(pole_angle)

        ddq = inv(M) @ (G - C @ dq + B * u)
        return np.array([cart_vel, pole_vel, ddq[0, 0], ddq[1, 0]])

    def linearize(self, x_eq, u_eq):
        cart_pos, pole_angle, cart_vel, pole_vel = x_eq

        M0    = self._M(pole_angle)
        B_mat = np.array([[1], [0]])

        dGdq = np.array([
            [0,                                                    0],
            [0, self.m_p * self.L * self.g * np.cos(pole_angle)]
        ])

        A_lin = np.block([
            [np.zeros((2, 2)), np.eye(2)       ],
            [inv(M0) @ dGdq,   np.zeros((2, 2))]
        ])
        B_lin = np.vstack([
            np.zeros((2, 1)),
            inv(M0) @ B_mat
        ])
        return A_lin, B_lin

    def discrete_dynamics(self, x, u, dt):
        # 미구현 (Euler 또는 ZOH)
        pass

    def f(self, state: 'CartpoleState', kernel: 'MujocoKernel', u):
        kernel.ctrl_tau = u
        kernel.step()
        kernel.update_state(state)
        return state.x


In [ ]:
from scipy.linalg import expm
import numpy as np

def discrete_dynamics_zoh(self, x_eq, u_eq, dt):
    Ac, Bc = self.linearize(x_eq, u_eq)
    nx = Ac.shape[0]
    nu = Bc.shape[1]

    M = np.zeros((nx + nu, nx + nu))
    M[:nx, :nx] = Ac
    M[:nx, nx:] = Bc
    M = M * dt
    
    M_exp = expm(M)
    Ad = M_exp[:nx, :nx]
    Bd = M_exp[:nx, nx:]
    
    return Ad, Bd 

##### CartpoleState — 상태 컨테이너

In [ ]:
class CartpoleState:
    NQ = 2  # [cart_pos, pole_angle]
    NV = 2  # [cart_vel, pole_vel]

    def __init__(self):
        self._q  = np.zeros(self.NQ)
        self._dq = np.zeros(self.NV)

        self.cart_pos   = self._q[0:1]
        self.pole_angle = self._q[1:2]
        self.cart_vel   = self._dq[0:1]
        self.pole_vel   = self._dq[1:2]

    @property
    def q(self):  return self._q
    @q.setter
    def q(self, val): np.copyto(self._q, val)

    @property
    def dq(self): return self._dq
    @dq.setter
    def dq(self, val): np.copyto(self._dq, val)

    @property
    def x(self):
        return np.concatenate([self._q, self._dq])  # [cart_pos, pole_angle, cart_vel, pole_vel]


##### MujocoKernel — 시뮬레이터 커널

In [ ]:
class MujocoKernel:
    def __init__(self, xml_path):
        self.model = mujoco.MjModel.from_xml_path(xml_path)
        self.data  = mujoco.MjData(self.model)

    def reset(self):
        if self.model.nkey > 0:
            mujoco.mj_resetDataKeyframe(self.model, self.data, 0)
        else:
            mujoco.mj_resetData(self.model, self.data)

    def update_state(self, state: CartpoleState):
        state.q  = self.data.qpos   # [cart_pos, pole_angle]
        state.dq = self.data.qvel   # [cart_vel, pole_vel]

    @property
    def ctrl_tau(self): return self.data.ctrl[0]
    @ctrl_tau.setter
    def ctrl_tau(self, u): self.data.ctrl[0] = u

    def step(self): mujoco.mj_step(self.model, self.data)


# 커널 & 상태 초기화
kernel = MujocoKernel(MJCF)
state  = CartpoleState()

kernel.reset()
kernel.update_state(state)

x_eq = state.x.copy()   # [cart_pos, pole_angle, cart_vel, pole_vel] @ 평형
u_eq = 0.0

In [ ]:
def LQR(A, B, Q, R):

    P = solve_continuous_are(A, B, Q, R)
    K = inv(R) @ B.T @ P

    return K

##### 시뮬레이션 초기화

In [ ]:
state  = CartpoleState()
kernel = MujocoKernel(MJCF)

mujoco.mj_resetData(kernel.model, kernel.data)
mujoco.mj_forward(kernel.model, kernel.data)

# MuJoCo에서 실제 파라미터 읽어오기
m_c  = kernel.model.body_mass[kernel.model.body('cart').id]
m_p  = kernel.model.body_mass[kernel.model.body('pole').id]
L    = 0.6   # XML: fromto='0 0 0 0 0 0.6'
gear = kernel.model.actuator_gear[0, 0]

print(f"m_c  = {m_c:.4f} kg")
print(f"m_p  = {m_p:.4f} kg")
print(f"L    = {L:.4f} m")
print(f"gear = {gear:.1f}")

dyn = CartPoleDynamics(m_c=m_c, m_p=m_p, L=L)

x_eq = np.array([0.0, 0.0, 0.0, 0.0])
u_eq = 0.0

A, B = dyn.linearize(x_eq, u_eq)
B = B * gear   # motor 타입: ctrl → force = ctrl * gear

print("A =\n", A)
print("B =\n", B)

# 로깅용 배열
times  = []
log_x  = []
log_u  = []


##### 시뮬레이션 루프

In [ ]:
import time

# 주기 설정
render_fps    = 60
render_dt     = 1.0 / render_fps
next_render_t = kernel.data.time + render_dt

ctrl_hz       = 200
ctrl_dt       = 1.0 / ctrl_hz
next_ctrl_t   = kernel.data.time

sim_duration  = 100.0   # 초

mujoco.mj_resetData(kernel.model, kernel.data)
kernel.data.qpos[0] = 0.0
kernel.data.qpos[1] = 0.1   # pole_angle: 수직 위(0rad) 기준 0.1rad 교란
kernel.data.qvel[:] = 0.0
mujoco.mj_forward(kernel.model, kernel.data)

wall_start = time.time()

Q = np.diag([100, 1, 1, 1])
R = np.array([[1.0]])

with mujoco.viewer.launch_passive(kernel.model, kernel.data,
                                  show_left_ui=False, show_right_ui=True) as viewer:

    K = LQR(A, B, Q, R)

    while viewer.is_running() and kernel.data.time < sim_duration:

        # 실시간 동기화
        sim_time  = kernel.data.time
        wall_time = time.time() - wall_start
        if sim_time > wall_time:
            time.sleep(sim_time - wall_time)

        # 제어 루프
        if kernel.data.time >= next_ctrl_t:
            kernel.update_state(state)
            x = state.x

            u = u_eq + float((-K @ (x - x_eq))[0])
            kernel.ctrl_tau = u

            # 로깅
            times.append(kernel.data.time)
            log_x.append(x.copy())
            log_u.append(u)

            next_ctrl_t += ctrl_dt

        kernel.step()

        # 렌더 루프
        if kernel.data.time >= next_render_t:
            viewer.sync()
            next_render_t += render_dt


##### 결과 시각화

In [ ]:
import matplotlib.pyplot as plt

times_arr = np.array(times)
log_x_arr = np.array(log_x)   # (N, 4)
log_u_arr = np.array(log_u)   # (N,)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

axes[0].plot(times_arr, log_x_arr[:, 0], label='cart_pos')
axes[0].set_ylabel('cart pos [m]')
axes[0].axhline(0, color='k', linestyle='--', linewidth=0.7)
axes[0].grid(True); axes[0].legend()

axes[1].plot(times_arr, log_x_arr[:, 1], label='pole_angle', color='orange')
axes[1].set_ylabel('pole angle [deg]')
axes[1].axhline(0, color='k', linestyle='--', linewidth=0.7)
axes[1].grid(True); axes[1].legend()

axes[2].plot(times_arr, log_u_arr, label='u (ctrl)', color='green')
axes[2].set_ylabel('control input')
axes[2].set_xlabel('Time [s]')
axes[2].axhline(0, color='k', linestyle='--', linewidth=0.7)
axes[2].grid(True); axes[2].legend()

plt.suptitle('Cartpole Simulation')
plt.tight_layout()
plt.show()
